# Week 59, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w059d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
it = iter([1, 2, 3, 4])
print(next(it), 3 in it, list(it))

**Predict before you run:** what will this print?



In [ ]:
def gen():
    print("a")
    yield 1
    print("b")
    yield 2

g = gen()
print("start")
print(next(g))

**Predict before you run:** what will this print?



In [ ]:
g = (x * x for x in range(5))
print(sum(g), sum(g), max((len(w) for w in []), default=-1))

**Predict before you run:** what will this print?



In [ ]:
def inner():
    yield 1
    return "done"

def outer():
    result = yield from inner()
    yield result

print(list(outer()))

**Predict before you run:** what will this print?



In [ ]:
from itertools import groupby, accumulate, islice, count
print([k for k, _ in groupby("aabbbac")], list(accumulate([1, 2, 3])), list(islice(count(5, -2), 4)))

### Exercise: Every path from the root

A tree is a nested list `[value, child, child, ...]` (a leaf is `[value]`). Write `root_paths(tree)`: a recursive generator using `yield from` that yields each root-to-leaf path as a list of values, left to right. Then `path_sums(tree, target)`: the paths whose values add up to the target, as a list.

In [ ]:
def root_paths(tree, prefix=()):
    ___

def path_sums(tree, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import types\ndef test_paths():\n    \"recursive generators\"\n    t = [5, [4, [11, [7], [2]]], [8, [13], [4, [1]]]]\n    assert list(root_paths(t)) == [[5, 4, 11, 7], [5, 4, 11, 2], [5, 8, 13], [5, 8, 4, 1]]\n    assert path_sums(t, 22) == [[5, 4, 11, 2]] and list(root_paths([3])) == [[3]]\n    assert isinstance(root_paths([1]), types.GeneratorType) and \"yield from\" in source")

### Exercise: Smallest sums, lazily

Two lists of numbers are sorted. Write `pair_sums(a, b)`: a generator yielding `(x, y)` pairs, one from each list, in increasing order of `x + y` (ties: the pair with the earlier position in a, then in b), lazily: a heap starts with `(a[0] + b[j], 0, j)` for every j and, each time a pair is taken, adds the next item of a for that j. Then `k_smallest(a, b, k)`: the first k pairs.

In [ ]:
import heapq

def pair_sums(a, b):
    ___

def k_smallest(a, b, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_pairs():\n    \"a heap inside a generator\"\n    assert k_smallest([1, 7, 11], [2, 4, 6], 3) == [(1, 2), (1, 4), (1, 6)]\n    assert k_smallest([1, 1, 2], [1, 2, 3], 2) == [(1, 1), (1, 1)] and k_smallest([], [1], 3) == []\n    rng = random.Random(1)\n    for _ in range(50):\n        a = sorted(rng.randint(0, 20) for _ in range(rng.randint(0, 6)))\n        b = sorted(rng.randint(0, 20) for _ in range(rng.randint(0, 6)))\n        allp = sorted(((x, y) for x in a for y in b), key=lambda p: p[0] + p[1])\n        got = list(pair_sums(a, b))\n        assert [x + y for x, y in got] == [x + y for x, y in allp] and sorted(got) == sorted(allp)\n    big = list(range(100_000))\n    assert k_smallest(big, big, 3) == [(0, 0), (0, 1), (1, 0)]")

### Exercise: Forge: a stream library

Write `Stream` to the specification above.

In [ ]:
import itertools
from collections import deque

class Stream:
    def __init__(self, iterable):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\n\ndef test_stages():\n    \"stages\"\n    s = Stream(range(10))\n    assert s.map(lambda x: x * x).filter(lambda x: x % 2).to_list() == [1, 9, 25, 49, 81]\n    assert Stream(\"abcdef\").skip(2).take(3).to_list() == [\"c\", \"d\", \"e\"]\n    assert Stream([1, 3, 5, 2, 7]).take_while(lambda x: x % 2).to_list() == [1, 3, 5]\n    assert Stream([1, 3, 5, 2, 7]).drop_while(lambda x: x % 2).to_list() == [2, 7]\n    assert Stream(range(7)).chunk(3).to_list() == [[0, 1, 2], [3, 4, 5], [6]]\n    assert Stream(range(5)).window(3).to_list() == [(0, 1, 2), (1, 2, 3), (2, 3, 4)]\n    assert Stream(\"aAbBa\").unique(str.lower).to_list() == [\"a\", \"b\"]\n    assert Stream(\"aabccc\").runs().to_list() == [(\"a\", [\"a\", \"a\"]), (\"b\", [\"b\"]), (\"c\", [\"c\", \"c\", \"c\"])]\n    assert Stream([\"ab\", \"c\"]).flat_map(list).to_list() == [\"a\", \"b\", \"c\"]\n    assert Stream(\"xy\").enumerate(1).to_list() == [(1, \"x\"), (2, \"y\")]\n    for bad in (lambda: Stream([1]).chunk(0), lambda: Stream([1]).window(0)):\n        try:\n            bad()\n            assert False\n        except ValueError:\n            pass\n\ndef test_sinks():\n    \"sinks\"\n    assert Stream([4, 5]).first() == 4 and Stream([]).first(None) is None\n    try:\n        Stream([]).first()\n        assert False\n    except ValueError:\n        pass\n    assert Stream(range(5)).count() == 5 and Stream(range(5)).sum() == 10\n    assert Stream([1, 2, 3, 4]).reduce(lambda a, b: a * b) == 24 and Stream([]).reduce(lambda a, b: a + b, 0) == 0\n    try:\n        Stream([]).reduce(max)\n        assert False\n    except ValueError:\n        pass\n    assert Stream.of(3, 1, 2).to_list() == [3, 1, 2]\n    assert Stream.lines(\"tea\\n\\n  \\nmilk\\n\").map(str.upper).to_list() == [\"TEA\", \"MILK\"]\n\ndef test_lazy():\n    \"laziness and reuse\"\n    pulled = []\n    def source():\n        for i in itertools.count():\n            pulled.append(i)\n            yield i\n    first_big = Stream(source()).map(lambda x: x * 10).filter(lambda x: x > 25).first()\n    assert first_big == 30 and pulled == [0, 1, 2, 3]\n    s = Stream(range(3))\n    assert s.to_list() == [0, 1, 2] and s.to_list() == [0, 1, 2], \"a stream over a list or range can be reused\"\n    endless = Stream(itertools.count()).window(2).map(sum).take(3).to_list()\n    assert endless == [1, 3, 5]\n    pipeline = Stream(itertools.count(1)).flat_map(lambda n: [n] * n).runs().map(lambda kv: (kv[0], len(kv[1]))).take(4).to_list()\n    assert pipeline == [(1, 1), (2, 2), (3, 3), (4, 4)]\n    assert isinstance(Stream([]).map(str), Stream)")

> **Why the stages don't run.** Every stage returns a Stream wrapping another lazy object, so building `Stream(data).map(f).filter(g).take(5)` costs nothing: it's a chain of generators waiting to be asked. Only a sink pulls, and it pulls one value at a time through the whole chain. That's why the laziness test can count exactly how many items of the source were read (four), and why an endless source is fine. It's also the design of Java's streams, of LINQ in C#, and of Python packages such as more-itertools, all built on the same protocol you met on Monday.

### Exercise: Forge bonus: merging logs

Several programs write logs with lines `"HH:MM:SS source message"`, each log already in time order. Write, lazily throughout:

- `parse_line(line)`: `(time, source, message)`, or `None` for a line without all three parts or whose time isn't exactly two digits, two digits, two digits separated by colons. `records(lines)`: a generator of the parsed records, skipping the rest.
- `merged(*logs)`: one stream of records from all the logs in time order, using `heapq.merge` with a `key` (equal times keep the order of the logs given).
- `seconds(time)`: seconds since midnight.
- `per_minute(records)`: a generator of `(minute, count)`, where minute is `"HH:MM"`, using `groupby` (the records arrive in time order).
- `busy_minutes(records, threshold)`: the list of minutes with at least `threshold` records.
- `quiet_gaps(records, limit)`: a generator of `(time_a, time_b)` for neighbouring records more than `limit` seconds apart (`pairwise`).

In [ ]:
import heapq
from itertools import groupby, pairwise

def parse_line(line):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, types\n\nWEB = [\"09:00:01 web start\", \"09:00:30 web GET /\", \"09:01:05 web GET /about\", \"bad\", \"09:03:00 web stop\"]\nDB = [\"09:00:02 db connect\", \"09:00:30 db query\", \"9:1:0 db broken time\", \"09:01:59 db query\"]\n\ndef test_parse():\n    \"parsing\"\n    assert parse_line(\"09:00:01 web start up\") == (\"09:00:01\", \"web\", \"start up\")\n    assert parse_line(\"bad\") is None and parse_line(\"9:00:01 web x\") is None and parse_line(\"09:00:01 web\") is None\n    assert isinstance(records(WEB), types.GeneratorType) and len(list(records(WEB))) == 4\n\ndef test_merge():\n    \"heapq.merge\"\n    times = [r[:2] for r in merged(WEB, DB)]\n    assert times == [(\"09:00:01\", \"web\"), (\"09:00:02\", \"db\"), (\"09:00:30\", \"web\"), (\"09:00:30\", \"db\"),\n                     (\"09:01:05\", \"web\"), (\"09:01:59\", \"db\"), (\"09:03:00\", \"web\")]\n    assert list(merged()) == [] and list(merged([], WEB))[0][0] == \"09:00:01\"\n\ndef test_minutes():\n    \"groupby and pairwise\"\n    assert list(per_minute(merged(WEB, DB))) == [(\"09:00\", 4), (\"09:01\", 2), (\"09:03\", 1)]\n    assert busy_minutes(merged(WEB, DB), 2) == [\"09:00\", \"09:01\"] and busy_minutes([], 1) == []\n    assert list(quiet_gaps(merged(WEB, DB), 30)) == [(\"09:00:30\", \"09:01:05\"), (\"09:01:05\", \"09:01:59\"), (\"09:01:59\", \"09:03:00\")]\n    assert seconds(\"01:02:03\") == 3723\n\ndef test_lazy():\n    \"endless logs\"\n    def ticker(source, start, step):\n        for t in itertools.count(start, step):\n            h, rest = divmod(t, 3600)\n            m, s = divmod(rest, 60)\n            yield f\"{h:02d}:{m:02d}:{s:02d} {source} tick\"\n    first = list(itertools.islice(merged(ticker(\"a\", 0, 7), ticker(\"b\", 3, 5)), 5))\n    assert [r[0] for r in first] == [\"00:00:00\", \"00:00:03\", \"00:00:07\", \"00:00:08\", \"00:00:13\"]\n    gen = per_minute(merged(ticker(\"a\", 0, 7), ticker(\"b\", 3, 5)))\n    assert next(gen) == (\"00:00\", 9 + 12)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [iterables and iterators](w059d1.html), [generators](w059d2.html), [generator expressions](w059d3.html), [yield from and pipelines](w059d4.html) or [itertools](w059d5.html).

## Mixed review

## Forge: a stream library

Write a class `Stream(iterable)` that wraps an iterable and offers **stages** (each returning a new `Stream`, without doing any work) and **sinks** (each pulling values through and returning a result). Every stage must stay lazy, so a `Stream` over an endless source works as long as a sink stops early. Use generators and `itertools` inside.

- `iter(stream)` iterates over the values (so a `Stream` works in `for`, `list` and so on; a stream over a list or range can be iterated again).
- Constructors: `Stream.of(*items)` and `Stream.lines(text)`: the non-blank lines of a string.
- Stages: `map(func)`, `filter(pred)`, `take(n)`, `skip(n)`, `take_while(pred)`, `drop_while(pred)`, `chunk(n)` (lists of n, the last possibly shorter), `window(n)` (tuples of n neighbours), `unique(key=None)` (first occurrences), `runs(key=None)` (pairs `(key, list_of_items)` for each run of neighbouring items with the same key), `flat_map(func)` (each item's iterable, flattened), `enumerate(start=0)`. `chunk` and `window` raise `ValueError` for n < 1.
- Sinks: `to_list()`, `first(default)` (the first value; with no default, `ValueError` on an empty stream), `count()`, `sum()`, `reduce(func, initial)` (fold left; with no initial, start from the first value, and `ValueError` if empty).

## Forge bonus: merging logs

## Weekly quiz

## Week 59 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can explain iter and next, tell iterables from iterators, and use iter with a sentinel. ([Day 1](w059d1.html))
- I can write generators, explain how they pause, and clean up with finally. ([Day 2](w059d2.html))
- I can write generator expressions, use them with any, next and sum, and know when their parts are evaluated. ([Day 3](w059d3.html))
- I can use yield from for recursive generators and return values, and build pipelines of stages. ([Day 4](w059d4.html))
- I can use itertools to slice, combine, group, accumulate and enumerate combinations. ([Day 5](w059d5.html))
- I built a stream library and a log merger. (Today)

Tomorrow is a **rest day**. Next week: **Functions in depth**: closures, `nonlocal` and decorators.